# Dataset connection check
Run the cell using your connected Colab kernel. It reads file metadata and five target/timing rows per dataset, exports feature definitions if available, and downloads connection_audit.json. Copy that file into D:\Scalewave so the local analysis agent can inspect it. This does not transfer the full datasets or grant remote execution access.

In [ ]:
from pathlib import Path
import json
import platform
import zipfile

ROOT = Path('/content/drive/MyDrive/Scalewave')
print('Runtime:', platform.platform())
print('Working directory:', Path.cwd())
if not ROOT.exists():
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except ImportError:
        raise RuntimeError('Select the connected Colab kernel, or change ROOT to the local dataset folder.')
if not ROOT.is_dir():
    raise FileNotFoundError(f'Dataset folder not accessible: {ROOT}')

import pyarrow.parquet as pq
report = {'root': str(ROOT), 'files': [], 'parquet': [], 'metadata': {}}
for path in sorted(ROOT.iterdir()):
    report['files'].append({'name': path.name, 'directory': path.is_dir(),
                            'bytes': path.stat().st_size if path.is_file() else None})
for path in sorted(ROOT.glob('BTCUSDT_signed_return*.parquet')):
    pf = pq.ParquetFile(path)
    names = pf.schema_arrow.names
    audit_cols = [c for c in names if c in (
        'decision_time', 'canonical_symbol', 'horizon_minutes',
        'signed_return', 'direction', 'label_status') or 'available_after' in c]
    batch = next(pf.iter_batches(columns=audit_cols, batch_size=5), None)
    report['parquet'].append({
        'name': path.name, 'rows': pf.metadata.num_rows,
        'row_groups': pf.num_row_groups, 'columns': names,
        'schema': str(pf.schema_arrow),
        'sample_timing_and_target_rows': batch.to_pylist() if batch else []})
archive = ROOT / 'HANDOFF_2_CANONICAL_XY_V1_1.zip'
if archive.exists():
    with zipfile.ZipFile(archive) as z:
        for info in z.infolist():
            if Path(info.filename).name in {'FEATURE_DICTIONARY.json', 'FEATURE_SCHEMAS.json', 'HANDOFF_2_QA.json', 'OUTPUT_FILES.json'}:
                if info.file_size <= 20_000_000:
                    report['metadata'][info.filename] = json.loads(z.read(info))
                else:
                    report['metadata'][info.filename] = {'skipped_bytes': info.file_size}
print('Files:', json.dumps(report['files'], indent=2))
print('Datasets:', [(p['name'], p['rows'], len(p['columns'])) for p in report['parquet']])
if len(report['parquet']) < 2:
    print('Expected both the large-move and smaller-move files; inspect the listing above.')
output = ROOT / 'connection_audit.json'
output.write_text(json.dumps(report, indent=2, default=str), encoding='utf-8')
print('Audit saved:', output)
try:
    from google.colab import files
    files.download(str(output))
except ImportError:
    print('Copy the audit JSON to D:/Scalewave for analysis here.')
